# 01 · Chart Makeover: from a default chart to a story chart
**Presentation & Data Storytelling with AI Tools** · Day 1 · Module 2 · about 25 minutes

You will take real world data (Gapminder: life expectancy, income and population for 142 countries, 1952–2007), make the chart most tools give you by default, then rebuild it as a chart that tells **one** story.

**How to use this notebook (no coding needed)**
1. Click a gray code cell, then press **Shift + Enter** to run it. Run the cells from top to bottom.
2. Only change lines marked with ✏️. Change the word or number in quotes, then run the cell again.
3. If something breaks, use **Runtime → Restart and run all**.

In [ ]:
# ▶ Load the tools and the data (Gapminder comes built into Colab's plotly library — no upload needed)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

gap = px.data.gapminder()
print(f"{gap['country'].nunique()} countries · years {gap['year'].min()}–{gap['year'].max()} · {len(gap):,} rows")
gap.head(8)

## Step 1 · The default chart
This is what you get when you hand data to a tool and accept every default: every country, every color, a legend to decode. Run it and ask yourself: **what is this chart trying to tell me?**

In [ ]:
# ▶ The "default" chart: all 25 countries in the Americas, rainbow colors, legend
americas = gap[gap["continent"] == "Americas"]

fig, ax = plt.subplots(figsize=(10, 5.6))
for country, d in americas.groupby("country"):
    ax.plot(d["year"], d["lifeExp"], label=country)
ax.set_title("Life expectancy by country")
ax.set_xlabel("year")
ax.set_ylabel("lifeExp")
ax.legend(ncol=3, fontsize=7)
plt.show()

**What's wrong?** Nothing is *incorrect*, but nothing is *said*:
- 25 lines fight for attention, so no line wins.
- The legend makes the reader match colors to names (your eyes jump back and forth).
- The title describes the data instead of stating a finding.
- Column names (`lifeExp`) are labels for computers, not people.

## Step 2 · Load the story style
One cell of settings removes chart junk from every chart after it: no top/right borders, light gridlines, left-aligned titles. It also adds two helpers: `add_titles()` and `add_source()`.

In [ ]:
# ▶ STORY STYLE — run once. Every chart after this cell gets the clean look.
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

NAVY, CORAL, GRAY, LIGHT, INK = "#1B2A41", "#FF6F59", "#BFC5CF", "#EEF1F5", "#4A5568"
plt.rcParams.update({
    "figure.figsize": (10, 5.6), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.spines.left": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": LIGHT, "grid.linewidth": 1,
    "axes.edgecolor": GRAY, "axes.labelcolor": INK, "axes.axisbelow": True,
    "xtick.color": INK, "ytick.color": INK, "ytick.left": False,
    "font.size": 11, "axes.titlesize": 16, "axes.titleweight": "bold",
})

def add_titles(ax, title, subtitle=""):
    """Headline (the insight) + a smaller subtitle (what is measured), left-aligned."""
    ax.set_title(title, loc="left", pad=34 if subtitle else 12, color=NAVY)
    if subtitle:
        ax.text(0, 1.035, subtitle, transform=ax.transAxes, fontsize=11, color=INK, va="bottom")

def add_source(fig, text):
    """Small source line in the bottom-left corner: always say where data came from."""
    fig.text(0.01, -0.02, text, fontsize=9, color="#8A94A6", ha="left", va="top")

print("Story style loaded ✔")

## Step 3 · Decide the story, then design for it
Storytellers pick the point **before** they draw. Here the point is a comparison between two countries. Everything else becomes quiet gray context.

✏️ Try the default first (Chile vs. United States), then swap in your own pair. The headline is written **from the data**, so it stays true whichever countries you pick.

In [ ]:
COUNTRY_A = "Chile"            # ✏️ the country your story is about
COUNTRY_B = "United States"    # ✏️ the country you compare it with
CONTINENT = "Americas"         # ✏️ gray background countries: Africa, Americas, Asia, Europe, Oceania

a = gap[gap["country"] == COUNTRY_A].sort_values("year")
b = gap[gap["country"] == COUNTRY_B].sort_values("year")
if a.empty or b.empty:
    raise ValueError("Check the spelling of COUNTRY_A / COUNTRY_B. Valid names: " + ", ".join(sorted(gap["country"].unique())))

first, last = int(a["year"].iloc[0]), int(a["year"].iloc[-1])
g0 = a["lifeExp"].iloc[0] - b["lifeExp"].iloc[0]     # gap at the start (positive = A ahead)
g1 = a["lifeExp"].iloc[-1] - b["lifeExp"].iloc[-1]   # gap at the end

def the(c):   # "the United States", "the Philippines" ... reads better in a headline
    return "the " + c if c.startswith(("United", "Central", "Dominican", "Czech", "Slovak")) or c in ("Netherlands", "Philippines", "Gambia", "Comoros") else c
A, B = the(COUNTRY_A), the(COUNTRY_B)

# The headline states the finding, computed from the data
if g0 < 0 <= g1:
    headline = f"{A[0].upper() + A[1:]} started {abs(g0):.0f} years behind {B} and caught up by {last}"
elif g1 <= 0 < g0:
    headline = f"{B[0].upper() + B[1:]} started {g0:.0f} years behind {A} and caught up by {last}"
elif abs(g1) < abs(g0):
    headline = f"The life-expectancy gap between {A} and {B} shrank from {abs(g0):.0f} to {abs(g1):.0f} years"
else:
    headline = f"The life-expectancy gap between {A} and {B} grew from {abs(g0):.0f} to {abs(g1):.0f} years"

fig, ax = plt.subplots()
for country, d in gap[gap["continent"] == CONTINENT].groupby("country"):        # 1) quiet context
    ax.plot(d["year"], d["lifeExp"], color=GRAY, lw=1, alpha=0.6, zorder=1)
ax.plot(b["year"], b["lifeExp"], color=NAVY, lw=3, zorder=3)                    # 2) the comparison
ax.plot(a["year"], a["lifeExp"], color=CORAL, lw=4, zorder=4)                   # 3) the hero
for d, color in [(a, CORAL), (b, NAVY)]:                                        # 4) direct labels, no legend
    ax.text(d["year"].iloc[-1] + 0.8, d["lifeExp"].iloc[-1], d["country"].iloc[0],
            color=color, fontweight="bold", va="center")
ax.annotate(f"{abs(g0):.0f}-year gap in {first}",                               # 5) point at the moment that matters
            xy=(first, (a["lifeExp"].iloc[0] + b["lifeExp"].iloc[0]) / 2),
            xytext=(first + 6, min(a["lifeExp"].iloc[0], b["lifeExp"].iloc[0]) - 6),
            arrowprops=dict(arrowstyle="->", color=INK), color=INK)
ax.set_xlim(first - 1, last + 9)
add_titles(ax, headline, f"Life expectancy at birth (years), {first}–{last}. Gray lines: other countries in {CONTINENT}.")
add_source(fig, "Source: Gapminder (gapminder.org), via the plotly.express sample data")
story_fig = fig     # saved so you can download it in Step 6
plt.show()

**The five moves you just saw** (use them on every chart you make):
1. **Gray is the default**, color is a highlighter. Only the story gets color.
2. **Label lines directly** at their ends. No legend to decode.
3. **The title is the finding** (an *action title*). The subtitle says what is measured.
4. **Annotate the moment** that matters.
5. **Cite the source** in small text. Trust is part of the story.

## Step 4 · Bars: sort them and highlight one
Alphabetical order is the computer's order. Sorted order is the reader's order: it answers "who's first, who's last, where's mine?" at a glance.

In [ ]:
YEAR = 2007                 # ✏️ any of 1952, 1957, ... 2007 (every 5 years)
BAR_CONTINENT = "Americas"  # ✏️
HIGHLIGHT = "Chile"         # ✏️ must be a country on that continent

snap = gap[(gap["year"] == YEAR) & (gap["continent"] == BAR_CONTINENT)]
if HIGHLIGHT not in set(snap["country"]):
    raise ValueError(f"{HIGHLIGHT} is not in {BAR_CONTINENT}. Try one of: " + ", ".join(sorted(snap['country'])))

fig, (left, right) = plt.subplots(1, 2, figsize=(14, 6.5))

# BEFORE: alphabetical, vertical, every bar the same color
left.bar(snap["country"], snap["lifeExp"])
left.set_title("Before: alphabetical", loc="left", color=INK)
left.tick_params(axis="x", rotation=90, labelsize=8)

# AFTER: sorted, horizontal (readable names), one highlight, one label
s = snap.sort_values("lifeExp")
rank = int(s["lifeExp"].rank(ascending=False)[s["country"] == HIGHLIGHT].iloc[0])
right.barh(s["country"], s["lifeExp"], color=[CORAL if c == HIGHLIGHT else GRAY for c in s["country"]])
val = s.loc[s["country"] == HIGHLIGHT, "lifeExp"].iloc[0]
right.text(val + 0.5, HIGHLIGHT, f"{val:.1f} yrs", va="center", color=CORAL, fontweight="bold")
right.grid(axis="y", visible=False); right.grid(axis="x", color=LIGHT)
right.tick_params(axis="y", labelsize=8)
add_titles(right, f"After: {HIGHLIGHT} ranks #{rank} of {len(s)} in {BAR_CONTINENT}", f"Life expectancy at birth, {YEAR}")
add_source(fig, "Source: Gapminder, via plotly.express sample data")
plt.tight_layout()
plt.show()

## Step 5 · The Rosling moment (interactive)
In 2006 Hans Rosling turned this exact dataset into one of the most-watched data talks ever ("The best stats you've ever seen", TED). Press **▶** under the chart and watch 50 years move. Hover a bubble to see the country.

As it plays, narrate it out loud the way Rosling did: *what moved, which way, and why it matters.* That narration is the story; the chart is the stage.

In [ ]:
# ▶ Animated bubble chart: x = income, y = life expectancy, size = population
fig = px.scatter(
    gap, x="gdpPercap", y="lifeExp", size="pop", color="continent",
    hover_name="country", animation_frame="year", animation_group="country",
    log_x=True, size_max=55, range_x=[100, 100000], range_y=[25, 90],
    labels={"gdpPercap": "Income per person (GDP per capita, US$, log scale)",
            "lifeExp": "Life expectancy (years)", "continent": "Continent"},
    title="Press ▶ — 50 years of health and wealth",
    template="simple_white", height=600,
)
fig.show()

## Step 6 · Download your story chart for your slides
This saves the Step 3 chart as a sharp PNG (good for PowerPoint, Google Slides, Gamma or Canva).

In [ ]:
FILENAME = "my_story_chart.png"   # ✏️ name your file

story_fig.savefig(FILENAME, dpi=200, bbox_inches="tight")
try:
    from google.colab import files
    files.download(FILENAME)      # your browser downloads the file
except ImportError:
    print(f"Saved {FILENAME} in this folder.")

## Challenges (pick one)
1. **New story.** Change `CONTINENT` to `"Asia"` and compare `"Korea, Rep."` with `"China"`. Does the headline still read well? Rewrite it by hand if you can say it better.
2. **AI pair-programmer.** Click the ✨ **Gemini** button in Colab (or copy the Step 3 cell into Claude/ChatGPT) and ask: *"Change this chart to show GDP per capita (gdpPercap) instead of life expectancy, keep the same style, and update the headline logic."* Run what it gives you. Did it work the first time? What did you have to fix?
3. **Say it in one sentence.** Under your chart, write the one sentence you would say out loud while this chart is on screen.